# Gaussian Process Light Curve Fitting for Supernovae

This notebook demonstrates how to fit supernova light curves using Gaussian Processes, save the results, and visualize the fits. It includes robust handling for empty data arrays in plotting routines.

## 1. Import Required Libraries and Set Paths

Import all necessary libraries and set up file paths and constants for the analysis.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt

import george
from george.kernels import ExpSquaredKernel, ConstantKernel, Matern32Kernel

# Set up paths
COCO_PATH = "/Users/ravkaur/Desktop/research/kilonova-SED/PyCoCo_templates/"
DATALC_PATH = COCO_PATH + "/Inputs/Photometry/4_LCs_late_extrapolated/"
DATASPEC_PATH = COCO_PATH + "/Inputs/Spectroscopy/"
DATAINFO_PATH = COCO_PATH + "Inputs/SNe_Info/"
FILTER_PATH = COCO_PATH + "Inputs/Filters/"
OUTPUT_DIR = COCO_PATH + "Outputs/"

import sys
sys.path.insert(0, COCO_PATH + 'what_the_flux/')
import what_the_flux as wtf

%matplotlib inline

## 2. Define Utility Functions

Helper functions for error conversions and other utilities.

In [ ]:
def err_to_log10(flux, err_flux):
    flux = np.array(flux, dtype=float)
    err_flux = np.array(err_flux, dtype=float)
    return 1. / np.log(10.) * err_flux / flux

def err_from_log10(logflux, logerr_flux):
    return np.log(10.) * 10 ** logflux * logerr_flux

## 3. Define SNPhotometryClass

Class for loading, clipping, fitting, and plotting supernova photometry data.